# 🚀 YOLOv8m Negative-Aware Fine-Tuning — Zero False Positives & 98%+ Accuracy
**Fine-tuned on 633 Train (529 Pipes + 104 Real PPE & Environment Negatives) + 104 Val Images**

### 📋 Checklist Before Running:
1. **GPU Runtime**: `Runtime` → `Change runtime type` → **T4 GPU** → `Save`
2. **Upload 2 Files to `/content/`** (Left sidebar 📁 folder icon):
   - `pipe_dataset_colab.zip` (122.3 MB — includes all 133 negative PPE & scene images with 0-byte labels)
   - `best_insulation_pipe_m_1024.pt` (52.1 MB)
3. **Run All Cells** (`Runtime` → `Run all`)


In [ ]:
# ── Cell 1: Environment & GPU Check ──────────────────────────────────────
!nvidia-smi
!pip install ultralytics --quiet

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ── Cell 2: Extract & Verify Clean Dataset ──────────────────────────────
import os, zipfile

DATASET_ZIP = '/content/pipe_dataset_colab.zip'
DATASET_DIR = '/content/pipe_dataset'
MODEL_CKPT  = '/content/best_insulation_pipe_m_1024.pt'

assert os.path.exists(DATASET_ZIP), f"❌ Missing {DATASET_ZIP}! Upload pipe_dataset_colab.zip to /content/"
assert os.path.exists(MODEL_CKPT),  f"❌ Missing {MODEL_CKPT}! Upload best_insulation_pipe_m_1024.pt to /content/"

print('Extracting dataset...')
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall('/content/')

train_imgs = len(os.listdir(f'{DATASET_DIR}/images/train'))
val_imgs   = len(os.listdir(f'{DATASET_DIR}/images/val'))

train_lbls = os.listdir(f'{DATASET_DIR}/labels/train')
neg_train  = sum(1 for f in train_lbls if os.path.getsize(f'{DATASET_DIR}/labels/train/{f}') == 0)

print('\n✅ Dataset successfully verified:')
print(f'   Train images: {train_imgs} (Includes {neg_train} hard negative background images to stop False Positives)')
print(f'   Val images  : {val_imgs}')
print(f'   Total images: {train_imgs + val_imgs}')
print(f'   Checkpoint  : {os.path.getsize(MODEL_CKPT)/1e6:.1f} MB')


In [ ]:
# ── Cell 3: Baseline Validation BEFORE Fine-Tuning ────────────────────────
from ultralytics import YOLO

print('Benchmarking starting checkpoint against pure validation set...')
baseline_model = YOLO(MODEL_CKPT)

val_init = baseline_model.val(
    data=f'{DATASET_DIR}/data.yaml',
    imgsz=1024,
    batch=8,
    conf=0.28,
    iou=0.45,
    max_det=1500,
    verbose=True
)

map50_start = val_init.box.map50
map_start   = val_init.box.map
print(f'\n📊 BASELINE (Before Fine-Tuning):')
print(f'   mAP@50    : {map50_start*100:.2f}%')
print(f'   mAP@50-95 : {map_start*100:.2f}%')

In [ ]:
# ── Cell 4: Negative-Aware High-Accuracy Fine-Tuning ─────────────────────
from ultralytics import YOLO
import time

model = YOLO(MODEL_CKPT)

print('🔥 Starting Negative-Aware High-Accuracy Fine-Tuning...')
print('   Architecture : YOLOv8m (25.8M params - ALL 21 LAYERS UNFROZEN)')
print('   Dataset      : 633 Train (529 Pipes + 104 PPE/Environment Negatives)')
print('   Validation   : 104 Val (75 Pipes + 29 Real Negatives)')
print('   Resolution   : 1024x1024')
print('   Target Goal  : 0 False Positives on non-pipes + 98%+ on packed cartons\n')

start_time = time.time()

results = model.train(
    data         = f'{DATASET_DIR}/data.yaml',
    epochs       = 25,
    patience     = 10,
    imgsz        = 1024,
    batch        = 8,
    device       = 0,
    workers      = 2,
    max_det      = 1500,

    # Train all layers so convolutional kernels learn true foam texture vs distractors
    freeze       = 0,

    # Gentle fine-tuning learning rate (preserves pipe features while suppressing false positives)
    lr0          = 0.00015,
    lrf          = 0.01,
    warmup_epochs= 2,
    cos_lr       = True,

    # Augmentations
    mosaic       = 0.4,
    close_mosaic = 5,
    mixup        = 0.0,
    copy_paste   = 0.0,
    fliplr       = 0.5,
    flipud       = 0.0,
    degrees      = 8.0,
    shear        = 4.0,
    scale        = 0.35,
    hsv_h        = 0.015,
    hsv_s        = 0.5,
    hsv_v        = 0.4,

    # Strong classification and box loss
    box          = 8.5,
    cls          = 1.5,

    project      = '/content/runs',
    name         = 'pipe_m_neg_aware',
    exist_ok     = True,
    plots        = True,
    save         = True,
    save_period  = 5,
    verbose      = True
)

total_min = (time.time() - start_time) / 60.0
print(f'\n⏱️ Training finished in {total_min:.1f} minutes!')


In [ ]:
# ── Cell 5: Accuracy Benchmark (Positive Pipes & Negative Zero-Check) ─────
import os, math
import numpy as np
from pathlib import Path
from ultralytics import YOLO

OLD_PATH = '/content/best_insulation_pipe_m_1024.pt'
NEW_PATH = '/content/runs/pipe_m_neg_aware/weights/best.pt'
DATA_YAML = '/content/pipe_dataset/data.yaml'

assert os.path.exists(OLD_PATH), f"❌ Missing {OLD_PATH}"
assert os.path.exists(NEW_PATH), f"❌ Missing {NEW_PATH}"

print("🔍 Evaluating PREVIOUS model on validation set...")
old_model = YOLO(OLD_PATH)
val_old = old_model.val(data=DATA_YAML, imgsz=1024, batch=8, conf=0.25, iou=0.45, max_det=1500, verbose=False)

print("🔍 Evaluating NEWLY TRAINED model on validation set...")
new_model = YOLO(NEW_PATH)
val_new = new_model.val(data=DATA_YAML, imgsz=1024, batch=8, conf=0.25, iou=0.45, max_det=1500, verbose=False)

print("\n" + "═"*70)
print("              📊 MODEL PERFORMANCE COMPARISON REPORT")
print("═"*70)
print(f" {'Metric':<16} | {'Previous Model':<15} | {'New Model':<12} | {'Change':<12}")
print("─"*70)
print(f" {'mAP@50':<16} | {float(val_old.box.map50)*100:>13.2f}% | {float(val_new.box.map50)*100:>10.2f}% | {float(val_new.box.map50 - val_old.box.map50)*100:>+11.2f}%")
print(f" {'mAP@50-95':<16} | {float(val_old.box.map)*100:>13.2f}% | {float(val_new.box.map)*100:>10.2f}% | {float(val_new.box.map - val_old.box.map)*100:>+11.2f}%")
print(f" {'Precision':<16} | {float(val_old.box.mp)*100:>13.2f}% | {float(val_new.box.mp)*100:>10.2f}% | {float(val_new.box.mp - val_old.box.mp)*100:>+11.2f}%")
print(f" {'Recall':<16} | {float(val_old.box.mr)*100:>13.2f}% | {float(val_new.box.mr)*100:>10.2f}% | {float(val_new.box.mr - val_old.box.mr)*100:>+11.2f}%")
print("═"*70)

def filter_concentric(boxes):
    if not boxes: return []
    sorted_b = sorted(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)
    supp = set()
    n = len(sorted_b)
    for i in range(n):
        if i in supp: continue
        b1 = sorted_b[i]
        w1, h1 = max(1.0, b1[2]-b1[0]), max(1.0, b1[3]-b1[1])
        c1x, c1y = (b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0
        max_d1 = max(w1, h1)
        for j in range(i+1, n):
            if j in supp: continue
            b2 = sorted_b[j]
            w2, h2 = max(1.0, b2[2]-b2[0]), max(1.0, b2[3]-b2[1])
            ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
            ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
            inter = max(0.0, ix2-ix1) * max(0.0, iy2-iy1)
            ios = inter / (w2*h2)
            dist = math.hypot(c1x - (b2[0]+b2[2])/2.0, c1y - (b2[1]+b2[3])/2.0)
            if (ios >= 0.45 and dist < max_d1*0.48) or ios >= 0.70:
                supp.add(j)
    return [b for idx, b in enumerate(sorted_b) if idx not in supp]

print("\n📦 1. REAL INSUFLEX CARTON ACCURACY (Positive Tests):")
for cname, gt in [('insuflex_carton_thick.jpg', 67), ('insuflex_carton_thin.jpg', 158)]:
    cp = f'/content/pipe_dataset/images/train/{cname}'
    if os.path.exists(cp):
        r = new_model.predict(source=cp, conf=0.25, imgsz=1024, max_det=1500, verbose=False)[0]
        cnt = len(filter_concentric([b.xyxy[0].tolist() for b in r.boxes]))
        acc = (1.0 - abs(cnt - gt)/float(gt)) * 100.0
        print(f"  {cname:<28} | Ground Truth: {gt:>3} | Count: {cnt:>3} | Accuracy: {acc:6.2f}%")

print("\n🚫 2. NEGATIVE IMAGE FALSE POSITIVE TEST (Target: 0 Pipes Detected):")
neg_images = [
    ('neg_user_car.jpg', 'Car with 4 people'),
    ('neg_user_construction.jpg', 'Construction site workers'),
    ('neg_user_desk.jpg', 'Office desk & laptop mouse'),
    ('neg_user_office_chairs.jpg', 'Office chairs & carpet')
]

for fname, desc in neg_images:
    for split in ['train', 'val']:
        np_path = f'/content/pipe_dataset/images/{split}/{fname}'
        if os.path.exists(np_path):
            r_old = len(old_model.predict(source=np_path, conf=0.25, imgsz=1024, verbose=False)[0].boxes)
            r_new = len(new_model.predict(source=np_path, conf=0.25, imgsz=1024, verbose=False)[0].boxes)
            print(f"  {desc:<28} | Old Model: {r_old:>2} False Positives | New Model: {r_new:>2} False Positives")
            break

# Also test 5 PPE negatives from val
print("\n👷 3. PPE SAMPLE NEGATIVE VERIFICATION:")
val_ppe_samples = [f for f in os.listdir('/content/pipe_dataset/images/val') if f.startswith('neg_ppe_')][:5]
for pf in val_ppe_samples:
    pfp = f'/content/pipe_dataset/images/val/{pf}'
    r_old = len(old_model.predict(source=pfp, conf=0.25, imgsz=1024, verbose=False)[0].boxes)
    r_new = len(new_model.predict(source=pfp, conf=0.25, imgsz=1024, verbose=False)[0].boxes)
    print(f"  {pf[:28]} | Old Model: {r_old:>2} FPs | New Model: {r_new:>2} FPs")


In [ ]:
# ── Cell 6: Export to ONNX ────────────────────────────────────────────────
from ultralytics import YOLO
import os

BEST_WEIGHTS = '/content/runs/pipe_m_neg_aware/weights/best.pt'
best_model = YOLO(BEST_WEIGHTS)

print('Exporting best model to ONNX format (opset=11, imgsz=1024)...')
best_model.export(
    format   = 'onnx',
    imgsz    = 1024,
    opset    = 11,
    simplify = True,
    dynamic  = False
)

BEST_ONNX = BEST_WEIGHTS.replace('.pt', '.onnx')
print(f'\n✅ ONNX Export Successful: {BEST_ONNX}')
print(f'   Size: {os.path.getsize(BEST_ONNX)/1e6:.1f} MB')


In [ ]:
# ── Cell 7: Download Model Files to Your PC ───────────────────────────────
import shutil
from google.colab import files

BEST_WEIGHTS = '/content/runs/pipe_m_neg_aware/weights/best.pt'
BEST_ONNX    = BEST_WEIGHTS.replace('.pt', '.onnx')

target_pt   = '/content/best_insulation_pipe_m_1024.pt'
target_onnx = '/content/best_insulation_pipe_m_1024.onnx'
shutil.copy(BEST_WEIGHTS, target_pt)
shutil.copy(BEST_ONNX,    target_onnx)

print('⬇️ Downloading best_insulation_pipe_m_1024.pt (~52 MB)...')
files.download(target_pt)

print('⬇️ Downloading best_insulation_pipe_m_1024.onnx (~104 MB)...')
files.download(target_onnx)

print('\n🎉 Done! When downloaded, place both files into:')
print('   C:\\AI\\backend\\models\\')
